In [1]:
# 1. Nạp dữ liệu gốc, kiểm tra schema và audit
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.utils import audit, check_disguised_missing
from src.pipeline import load_raw_data, clean_transactions

df_raw = load_raw_data(ROOT / "data/raw")
print(f"Kích thước dữ liệu gốc: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
display(audit(df_raw).sort_values("pct_missing", ascending=False))
display(check_disguised_missing(df_raw))

Nguồn: online_retail_II.csv | 1,067,371 dòng
Kích thước dữ liệu gốc: 1,067,371 dòng, 8 cột


,dtype,n_missing,pct_missing,n_unique,sample
Customer ID,float64,243007,22.77,5942,13085.0
Description,str,4382,0.41,5698,15CM CHRISTMAS GLASS BALL 20 LIGHTS
StockCode,str,0,0.00,5305,85048
Invoice,str,0,0.00,53628,489434
Quantity,int64,0,0.00,1057,12
InvoiceDate,datetime64[us],0,0.00,47635,2009-12-01 07:45:00
Price,float64,0,0.00,2807,6.95
Country,str,0,0.00,43,United Kingdom


,column,n_disguised,examples
0,Description,342,"[damaged, ?, check, ??]"
1,Country,756,[Unspecified]


In [2]:
# 2. Kiểm tra trước làm sạch
print(f"Số dòng trùng hoàn toàn (dữ liệu gốc): {df_raw.duplicated().sum():,}")
print(f"Số dòng thiếu Customer ID: {df_raw['Customer ID'].isna().sum():,}")
print(f"Phạm vi ngày: {df_raw['InvoiceDate'].min()} - {df_raw['InvoiceDate'].max()}")
# Chưa đủ thông tin để xác định cơ chế MAR/MNAR hoặc nguồn gốc từng dòng trùng.

Số dòng trùng hoàn toàn (dữ liệu gốc): 34,335
Số dòng thiếu Customer ID: 243,007
Phạm vi ngày: 2009-12-01 07:45:00 - 2011-12-09 12:50:00


## W3 – Chất lượng dữ liệu và sai lệch chọn mẫu
Thang điểm nội bộ: các proxy tỷ lệ ≥99%, ≥95%, ≥90%, ≥80%, thấp hơn lần lượt 5, 4, 3, 2, 1. Accuracy không có đối chứng nên chỉ đánh giá mức bằng chứng; Timeliness xét tính hiện hành, không phủ nhận giá trị lịch sử. Không dùng điểm trung bình để suy ra xác suất dữ liệu đúng.
So sánh nhóm có/thiếu ID trên dòng giao dịch đã chuẩn hóa, bỏ trùng, trước loại Customer ID; giá trị mua chỉ gồm dòng sản phẩm dương hợp lệ, chưa trừ hoàn trả. Khác biệt quan sát không xác định được MAR/MNAR hay số khách thiếu ID.

In [3]:
from src.data_quality import quality_dimensions, missing_id_comparison, invoice_timestamp_audit
quality = quality_dimensions(df_raw)
missing_summary, missing_by_month, missing_by_country = missing_id_comparison(df_raw)
display(quality)
display(missing_summary.round(4))
display(missing_by_month.round(4))
display(missing_by_country.sort_values('total', ascending=False).round(4))
print(f"Tỷ trọng gross purchase value thuộc dòng thiếu ID: {missing_summary.loc['Missing_ID', 'gross_value_gbp_share']:.2%}")
print('Tỷ trọng này đo phần giá trị giao dịch bị loại khỏi phân tích khách hàng, không phải tỷ lệ khách mất.')

,Dimension,Score_1_5,Evidence,Limitation
0,Completeness,4,97.09% ô không thiếu thật/giả,Tính theo ô; Customer ID cần xem riêng.
1,Accuracy,2,Không có dữ liệu đối chứng giao dịch thực,"2/5 là mức bằng chứng hạn chế, không phải tỷ l..."
2,Consistency,4,95.01% ô chuỗi không thừa khoảng trắng,Proxy hẹp; kiểm tra timestamp hóa đơn riêng ở W4.
3,Validity,5,99.42% dòng thỏa giá > 0 và dấu quantity,Gồm cả điều chỉnh/phi sản phẩm nên không đồng ...
4,Uniqueness,4,96.78% dòng không trùng hoàn toàn sau dòng đầu,Không xác minh được bản ghi trùng là giao dịch...
5,Timeliness,2,2009-12-01 07:45:00 → 2011-12-09 12:50:00,2/5 cho tính hiện hành (dữ liệu lịch sử 2009–2...


,all_lines,purchase_lines,gross_value_gbp,median_line_gbp,median_quantity,median_unit_price_gbp,all_lines_share,purchase_lines_share,gross_value_gbp_share
id_status,,,,,,,,,
Known_ID,797885,776844,1.708113e+07,12.48,6.0,1.95,0.7724,0.774,0.8689
Missing_ID,235151,226857,2.577915e+06,5.02,1.0,3.29,0.2276,0.226,0.1311


id_status,Known_ID,Missing_ID,total,missing_share
month,,,,
2009-12,30215,13122,43337,0.3028
2010-01,21375,8809,30184,0.2918
2010-02,22964,4858,27822,0.1746
2010-03,31656,7965,39621,0.2010
2010-04,26728,5951,32679,0.1821
2010-05,28156,5079,33235,0.1528
2010-06,30604,7618,38222,0.1993
2010-07,26558,5402,31960,0.1690
2010-08,25959,6015,31974,0.1881


id_status,Known_ID,Missing_ID,total,missing_share
Country,,,,
United Kingdom,699684,224036,923720,0.2425
EIRE,15529,1583,17112,0.0925
Germany,15788,0,15788,0.0000
France,13036,128,13164,0.0097
Netherlands,4981,0,4981,0.0000
Spain,3560,0,3560,0.0000
Switzerland,2951,117,3068,0.0381
Belgium,2911,0,2911,0.0000
Portugal,2295,114,2409,0.0473


Tỷ trọng gross purchase value thuộc dòng thiếu ID: 13.11%
Tỷ trọng này đo phần giá trị giao dịch bị loại khỏi phân tích khách hàng, không phải tỷ lệ khách mất.


In [4]:
# 3. Làm sạch bằng đúng các hàm trong pipeline (khởi tạo lại log mỗi lần chạy)
df, base_log = clean_transactions(df_raw)
print(f"Sau làm sạch: {len(df):,} dòng")
display(base_log)

Sau làm sạch: 794,437 dòng


,Step,Column,Finding,Action,Rows,Why
0,1.1,all_columns,"Tên cột chưa đồng nhất, InvoiceDate dạng chuỗi","Chuẩn hóa snake_case, ép kiểu datetime và loại...",1067371,Đảm bảo tính nhất quán khi trích xuất thời gia...
1,1.2,all,"34,335 dòng trùng lặp hoàn toàn","Xóa các dòng trùng lặp, giữ lại bản ghi đầu tiên",34335,Giả định các dòng giống hệt là trùng bản ghi; ...
2,1.3,customer_id,"235,151 dòng thiếu mã khách hàng",Loại bỏ các giao dịch không định danh được khá...,235151,Đơn vị phân tích là hành vi từng khách hàng
3,1.4,description,0 dòng thiếu hoặc chứa ký tự lỗi,Gán nhãn 'UNKNOWN_PRODUCT' thay vì xóa dòng,0,Giữ lại lịch sử chi tiêu và tần suất đơn hàng
4,1.5,country,582 dòng chưa xác định; khôi phục 0,Điền theo quốc gia đã biết phổ biến nhất của c...,0,Rows = số dòng thực sự được khôi phục
5,1.5a,country,582 dòng vẫn chưa xác định,Giữ nguyên quốc gia gốc chưa xác định; gán Unk...,582,"Rows = số dòng chưa khôi phục tại bước này, tr..."
6,1.6,stockcode,"3,387 dòng phí vận chuyển/điều chỉnh kế toán",Loại bỏ khỏi bảng giao dịch sản phẩm,3387,Không phản ánh hành vi chọn mua sản phẩm
7,1.7,invoice,"17,593 dòng đơn hủy/hoàn trả",Giữ nguyên và gắn cờ is_cancelled=True,17593,Tín hiệu quan trọng phản ánh hành vi hoàn đơn
8,1.8,price & quantity,61 dòng giá <= 0 hoặc mâu thuẫn dấu số lượng,Loại bỏ bản ghi bất khả thi,61,Tránh làm sai lệch doanh thu và giá trị giỏ hàng


In [5]:
# 4. Chẩn đoán ngoại lai theo IQR; không xóa chỉ vì nằm ngoài ngưỡng
# Tách mua và hoàn trả vì dấu total_amount khác nhau.
outlier_rows = []
for label, group in df.groupby("is_cancelled"):
    q1, q3 = group["total_amount"].quantile([0.25, 0.75])
    iqr = q3 - q1
    count = ((group["total_amount"] < q1 - 1.5 * iqr) | (group["total_amount"] > q3 + 1.5 * iqr)).sum()
    outlier_rows.append({"Step": "IQR_cancel" if label else "IQR_purchase", "Column": "total_amount",
                         "Finding": f"{count:,} dòng ngoài 1.5*IQR", "Action": "Giữ lại; kiểm tra theo nghiệp vụ",
                         "Rows": int(count), "Why": "Ngoại lai thống kê không nhất thiết là lỗi hoặc khách mua buôn"})
log_df = pd.concat([base_log, pd.DataFrame(outlier_rows)], ignore_index=True)
display(pd.DataFrame(outlier_rows))

,Step,Column,Finding,Action,Rows,Why
0,IQR_purchase,total_amount,"62,442 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,62442,Ngoại lai thống kê không nhất thiết là lỗi hoặ...
1,IQR_cancel,total_amount,"1,854 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,1854,Ngoại lai thống kê không nhất thiết là lỗi hoặ...


In [6]:
# 5. Tái kiểm định và lưu dữ liệu W4
assert df["customer_id"].notna().all()
assert np.isfinite(df[["price", "quantity", "total_amount"]].to_numpy()).all()
assert (df["price"] > 0).all()
assert (df.loc[~df["is_cancelled"], "quantity"] > 0).all()
assert (df.loc[df["is_cancelled"], "quantity"] < 0).all()
assert np.allclose(df["total_amount"], df["price"] * df["quantity"])
assert log_df["Step"].is_unique
print("=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH ===")
display(audit(df))
display(check_disguised_missing(df))
display(log_df)
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)
log_df.to_csv(ROOT / "reports/cleaning_log_w4.csv", index=False)
df.to_parquet(ROOT / "data/processed/retail_cleaned_w4.parquet", index=False)
print(f"Số dòng sạch: {len(df):,}; chạy python -m src.pipeline ở thư mục gốc trước notebook 02/03.")

=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH ===


,dtype,n_missing,pct_missing,n_unique,sample
invoice,str,0,0.0,43887,489434
stockcode,str,0,0.0,4636,85048
description,str,0,0.0,5246,15CM CHRISTMAS GLASS BALL 20 LIGHTS
quantity,int64,0,0.0,639,12
price,float64,0,0.0,477,6.95
customer_id,str,0,0.0,5876,13085
country,str,0,0.0,41,United Kingdom
invoice_date,datetime64[us],0,0.0,40630,2009-12-01 07:45:00
country_original,str,0,0.0,41,United Kingdom
is_cancelled,bool,0,0.0,2,False


,column,n_disguised,examples
0,country,519,[Unspecified]
1,country_original,519,[Unspecified]


,Step,Column,Finding,Action,Rows,Why
0,1.1,all_columns,"Tên cột chưa đồng nhất, InvoiceDate dạng chuỗi","Chuẩn hóa snake_case, ép kiểu datetime và loại...",1067371,Đảm bảo tính nhất quán khi trích xuất thời gia...
1,1.2,all,"34,335 dòng trùng lặp hoàn toàn","Xóa các dòng trùng lặp, giữ lại bản ghi đầu tiên",34335,Giả định các dòng giống hệt là trùng bản ghi; ...
2,1.3,customer_id,"235,151 dòng thiếu mã khách hàng",Loại bỏ các giao dịch không định danh được khá...,235151,Đơn vị phân tích là hành vi từng khách hàng
3,1.4,description,0 dòng thiếu hoặc chứa ký tự lỗi,Gán nhãn 'UNKNOWN_PRODUCT' thay vì xóa dòng,0,Giữ lại lịch sử chi tiêu và tần suất đơn hàng
4,1.5,country,582 dòng chưa xác định; khôi phục 0,Điền theo quốc gia đã biết phổ biến nhất của c...,0,Rows = số dòng thực sự được khôi phục
5,1.5a,country,582 dòng vẫn chưa xác định,Giữ nguyên quốc gia gốc chưa xác định; gán Unk...,582,"Rows = số dòng chưa khôi phục tại bước này, tr..."
6,1.6,stockcode,"3,387 dòng phí vận chuyển/điều chỉnh kế toán",Loại bỏ khỏi bảng giao dịch sản phẩm,3387,Không phản ánh hành vi chọn mua sản phẩm
7,1.7,invoice,"17,593 dòng đơn hủy/hoàn trả",Giữ nguyên và gắn cờ is_cancelled=True,17593,Tín hiệu quan trọng phản ánh hành vi hoàn đơn
8,1.8,price & quantity,61 dòng giá <= 0 hoặc mâu thuẫn dấu số lượng,Loại bỏ bản ghi bất khả thi,61,Tránh làm sai lệch doanh thu và giá trị giỏ hàng
9,IQR_purchase,total_amount,"62,442 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,62442,Ngoại lai thống kê không nhất thiết là lỗi hoặ...


Số dòng sạch: 794,437; chạy python -m src.pipeline ở thư mục gốc trước notebook 02/03.


## W4 – Kiểm tra hóa đơn có nhiều timestamp
Khóa là (customer_id, invoice), chỉ xét đơn mua hợp lệ. Kiểm tra span, hóa đơn qua ngày và qua cutoff/horizon; không mặc định xóa. W11 cắt dòng theo timestamp gốc trước khi gom đơn. Không in mã khách hàng/hóa đơn vào output báo cáo.

In [7]:
invoice_counts, multi_invoice, boundary_counts = invoice_timestamp_audit(df)
display(invoice_counts.to_frame('count'))
display(multi_invoice['span_minutes'].describe().to_frame('span_minutes'))
display(boundary_counts)
print('Hóa đơn nhiều timestamp không tự động là lỗi; cần đối chiếu nguồn nếu muốn thay đổi hoặc loại bỏ.')

,count
purchase_invoices,36600
multi_timestamp_invoices,64
cross_calendar_day,0
invoice_codes_shared_by_customers,0


,span_minutes
count,64.000000
mean,1.015625
std,0.125000
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


,cutoff,boundary,kind,crossing_invoices
0,2011-03-01,2011-03-01,cutoff,0
1,2011-03-01,2011-05-30,label_end,0
2,2011-06-01,2011-06-01,cutoff,0
3,2011-06-01,2011-08-30,label_end,0
4,2011-09-01,2011-09-01,cutoff,0
5,2011-09-01,2011-11-30,label_end,0


Hóa đơn nhiều timestamp không tự động là lỗi; cần đối chiếu nguồn nếu muốn thay đổi hoặc loại bỏ.
